In [6]:
%cd /content
!rm -rf ELEC475-Group22
!git clone https://github.com/madobri/ELEC475-Group22.git
%cd /content/ELEC475-Group22/Lab2
!pip install -q thop torchinfo
!pwd

/content
Cloning into 'ELEC475-Group22'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 73 (delta 0), reused 0 (delta 0), pack-reused 72 (from 4)
Receiving objects: 100% (73/73), 92.46 MiB | 34.43 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/ELEC475-Group22/Lab2
/content/ELEC475-Group22/Lab2


In [9]:
!mv /content/ELEC475-Group22/checkpoints/ /content/ELEC475-Group22/Lab2

In [10]:
!python -c "from utils.dataset import download_and_extract_tiny_imagenet as d; d('./data')"
!ls data/tiny-imagenet-200/val | head
!head -3 data/tiny-imagenet-200/val/val_annotations.txt

Download complete.
Extracting ./data/tiny-imagenet-200.zip...
Extraction complete.
images
val_annotations.txt
val_0.JPEG	n03444034	0	32	44	62
val_1.JPEG	n04067472	52	55	57	59
val_2.JPEG	n04070727	4	0	60	55


In [16]:
!mkdir -p results
!python train.py --model alexnet --norm bn --dataset tiny_imagenet --epochs 10 --save-model checkpoints/alexnet_tiny_imagenet.pth --plot-file plots/loss_tiny_imagenet.png 2>&1 | tee results/alexnet_tiny_imagenet.txt

ELEC 475: Lab 2 Training Engine
Device        : cuda (Tesla T4)
Model         : ALEXNET (Norm: BN)
Dataset       : TINY_IMAGENET (Augmentation: True)
Hyperparams   : Epochs=10, BatchSize=128, LR=0.001, Seed=42
Early Stopping: Disabled

Initializing DataLoaders...
Train batches: 781 (100000 samples), Val batches: 79, Classes: 200
Computational Complexity Profile: ALEXNET (bn)
Input Tensor Shape   : (1, 3, 64, 64)
------------------------------------------------------------
Total Parameters     : 6,806,152 (6.8062 M)
Trainable Parameters : 6,806,152
Multiply-Accumulates : 219,177,728 (219.18 MMACs)
Floating Point Ops   : 438,355,456 (438.36 MFLOPs)
Weight Size (FP32)   : 25.96 MB

Detailed Layer Specification:
Layer (type:depth-idx)                   Output Shape              Param #
AlexNet64                                [1, 200]                  --
├─Sequential: 1-1                        [1, 256, 3, 3]            --
│    └─Conv2d: 2-1                       [1, 64, 32, 32]           

In [18]:
!python compare_models.py --dataset cifar10 2>&1 | tee results/compare_models_cifar10.txt

ELEC 475: Lab 2 - Comprehensive Model Complexity & FLOPs Benchmark
Device        : cuda (Tesla T4)
Dataset       : CIFAR10 (10 classes)
Input Shape   : (1, 3, 64, 64)

Instantiating models for profiling...
Profiling FLOPs, Multiply-Accumulates, and Latency...

| Model Architecture          | Params (M) | MACs (M) | FLOPs (M) | Latency (ms) | Throughput (img/s) |
| ----------------------------|------------|----------|-----------|--------------|------------------- |
| 1. LeNet-5 (Base)           | 0.338      | 3.59     | 7.17      | 0.35         | 2897.5             |
| 2. LeNet-Modern (ReLU+Pool) | 0.446      | 4.59     | 9.19      | 0.35         | 2861.4             |
| 3. AlexNet-64 (None)        | 6.609      | 218.35   | 436.70    | 1.89         | 527.9              |
| 4. AlexNet-64 (LRN)         | 6.609      | 218.35   | 436.70    | 1.67         | 598.1              |
| 5. AlexNet-64 (BatchNorm)   | 6.611      | 218.98   | 437.97    | 1.48         | 674.8              |
| 6. Torchv

In [24]:
!python -m utils.visualization --plot-ablation

<frozen runpy>:130: RuntimeWarning: 'utils.visualization' found in sys.modules after import of package 'utils', but prior to execution of 'utils.visualization'; this may result in unpredictable behaviour


In [25]:
!python train.py --model alexnet --norm bn --dataset cifar10 --epochs 10 --no-augment --seed 42 --lr 3e-4 --plot-file plots/loss_alexnet_bn_lr3e-4.png 2>&1 | tee results/ablation_lr3e-4.txt

ELEC 475: Lab 2 Training Engine
Device        : cuda (Tesla T4)
Model         : ALEXNET (Norm: BN)
Dataset       : CIFAR10 (Augmentation: False)
Hyperparams   : Epochs=10, BatchSize=128, LR=0.0003, Seed=42
Early Stopping: Disabled

Initializing DataLoaders...
Train batches: 390 (50000 samples), Val batches: 79, Classes: 10
Computational Complexity Profile: ALEXNET (bn)
Input Tensor Shape   : (1, 3, 64, 64)
------------------------------------------------------------
Total Parameters     : 6,611,402 (6.6114 M)
Trainable Parameters : 6,611,402
Multiply-Accumulates : 218,983,168 (218.98 MMACs)
Floating Point Ops   : 437,966,336 (437.97 MFLOPs)
Weight Size (FP32)   : 25.22 MB

Detailed Layer Specification:
Layer (type:depth-idx)                   Output Shape              Param #
AlexNet64                                [1, 10]                   --
├─Sequential: 1-1                        [1, 256, 3, 3]            --
│    └─Conv2d: 2-1                       [1, 64, 32, 32]           4,864


In [22]:
!python train.py --model zoo_alexnet --dataset cifar10 --epochs 10 --no-augment --seed 42 --save-model checkpoints/zoo_alexnet.pth --plot-file plots/loss_zoo_alexnet.png 2>&1 | tee results/zoo_alexnet_cifar10.txt


ELEC 475: Lab 2 Training Engine
Device        : cuda (Tesla T4)
Model         : ZOO_ALEXNET (Norm: N/A)
Dataset       : CIFAR10 (Augmentation: False)
Hyperparams   : Epochs=10, BatchSize=128, LR=0.001, Seed=42
Early Stopping: Disabled

Initializing DataLoaders...
100%|██████████| 170M/170M [53:07<00:00, 53.5kB/s] 
Train batches: 390 (50000 samples), Val batches: 79, Classes: 10
Computational Complexity Profile: ZOO_ALEXNET (bn)
Input Tensor Shape   : (1, 3, 64, 64)
------------------------------------------------------------
Total Parameters     : 57,044,810 (57.0448 M)
Trainable Parameters : 57,044,810
Multiply-Accumulates : 94,099,392 (94.10 MMACs)
Floating Point Ops   : 188,198,784 (188.20 MFLOPs)
Weight Size (FP32)   : 217.61 MB

Detailed Layer Specification:
Layer (type:depth-idx)                   Output Shape              Param #
AlexNet                                  [1, 10]                   --
├─Sequential: 1-1                        [1, 256, 1, 1]            --
│    └─Conv

In [23]:
!python train.py --model alexnet --norm bn --dataset cifar10 --epochs 10 --no-augment --seed 123 --plot-file plots/loss_alexnet_bn_s123.png 2>&1 | tee results/alexnet_bn_seed123.txt
!python train.py --model alexnet --norm bn --dataset cifar10 --epochs 10 --no-augment --seed 999 --plot-file plots/loss_alexnet_bn_s999.png 2>&1 | tee results/alexnet_bn_seed999.txt

ELEC 475: Lab 2 Training Engine
Device        : cuda (Tesla T4)
Model         : ALEXNET (Norm: BN)
Dataset       : CIFAR10 (Augmentation: False)
Hyperparams   : Epochs=10, BatchSize=128, LR=0.001, Seed=123
Early Stopping: Disabled

Initializing DataLoaders...
Train batches: 390 (50000 samples), Val batches: 79, Classes: 10
Computational Complexity Profile: ALEXNET (bn)
Input Tensor Shape   : (1, 3, 64, 64)
------------------------------------------------------------
Total Parameters     : 6,611,402 (6.6114 M)
Trainable Parameters : 6,611,402
Multiply-Accumulates : 218,983,168 (218.98 MMACs)
Floating Point Ops   : 437,966,336 (437.97 MFLOPs)
Weight Size (FP32)   : 25.22 MB

Detailed Layer Specification:
Layer (type:depth-idx)                   Output Shape              Param #
AlexNet64                                [1, 10]                   --
├─Sequential: 1-1                        [1, 256, 3, 3]            --
│    └─Conv2d: 2-1                       [1, 64, 32, 32]           4,864


In [26]:
!python train.py --model alexnet --norm bn --dataset cifar10 --epochs 10 --no-augment --seed 42 --batch-size 64 --plot-file plots/loss_alexnet_bn_bs64.png 2>&1 | tee results/ablation_bs64.txt

ELEC 475: Lab 2 Training Engine
Device        : cuda (Tesla T4)
Model         : ALEXNET (Norm: BN)
Dataset       : CIFAR10 (Augmentation: False)
Hyperparams   : Epochs=10, BatchSize=64, LR=0.001, Seed=42
Early Stopping: Disabled

Initializing DataLoaders...
Train batches: 781 (50000 samples), Val batches: 157, Classes: 10
Computational Complexity Profile: ALEXNET (bn)
Input Tensor Shape   : (1, 3, 64, 64)
------------------------------------------------------------
Total Parameters     : 6,611,402 (6.6114 M)
Trainable Parameters : 6,611,402
Multiply-Accumulates : 218,983,168 (218.98 MMACs)
Floating Point Ops   : 437,966,336 (437.97 MFLOPs)
Weight Size (FP32)   : 25.22 MB

Detailed Layer Specification:
Layer (type:depth-idx)                   Output Shape              Param #
AlexNet64                                [1, 10]                   --
├─Sequential: 1-1                        [1, 256, 3, 3]            --
│    └─Conv2d: 2-1                       [1, 64, 32, 32]           4,864
│

In [27]:
!python demo_transforms.py 2>&1 | tee results/demo_transforms.txt

ELEC 475: Transform Paradigm & Variable Type Diagnostic

[1] OpenCV (cv2) Output:
    - Object Type      : <class 'numpy.ndarray'>
    - Data Type (dtype): float32
    - Shape (Layout)   : (64, 64, 3)  --> [Height, Width, Channels] (Channels-Last)
    - Value Range      : Min = 0.0000, Max = 1.0000
    - Color Space      : Converted BGR -> RGB
    - Memory Location  : Host System RAM (CPU)

[2] NumPy Output:
    - Object Type      : <class 'numpy.ndarray'>
    - Data Type (dtype): float32
    - Shape (Layout)   : (64, 64, 3)  --> [Height, Width, Channels] (Channels-Last)
    - Value Range      : Min = 0.0000, Max = 1.0000
    - Memory Location  : Host System RAM (CPU)

[3] PyTorch (torchvision) Tensor Output:
    - Object Type      : <class 'torch.Tensor'>
    - Data Type (dtype): torch.float32
    - Shape (Layout)   : torch.Size([3, 64, 64])  --> [Channels, Height, Width] (Channels-First)
    - Value Range      : Min = -1.9895, Max = 2.1265 (Standardized)
    - Memory Location  : cuda

In [28]:
!python demo_transforms.py 2>&1 | tee results/demo_transforms.txt

ELEC 475: Transform Paradigm & Variable Type Diagnostic

[1] OpenCV (cv2) Output:
    - Object Type      : <class 'numpy.ndarray'>
    - Data Type (dtype): float32
    - Shape (Layout)   : (64, 64, 3)  --> [Height, Width, Channels] (Channels-Last)
    - Value Range      : Min = 0.0000, Max = 1.0000
    - Color Space      : Converted BGR -> RGB
    - Memory Location  : Host System RAM (CPU)

[2] NumPy Output:
    - Object Type      : <class 'numpy.ndarray'>
    - Data Type (dtype): float32
    - Shape (Layout)   : (64, 64, 3)  --> [Height, Width, Channels] (Channels-Last)
    - Value Range      : Min = 0.0000, Max = 1.0000
    - Memory Location  : Host System RAM (CPU)

[3] PyTorch (torchvision) Tensor Output:
    - Object Type      : <class 'torch.Tensor'>
    - Data Type (dtype): torch.float32
    - Shape (Layout)   : torch.Size([3, 64, 64])  --> [Channels, Height, Width] (Channels-First)
    - Value Range      : Min = -1.9895, Max = 2.1265 (Standardized)
    - Memory Location  : cuda

In [29]:
!python -m utils.eda --dataset cifar10 --out-dir plots

<frozen runpy>:130: RuntimeWarning: 'utils.eda' found in sys.modules after import of package 'utils', but prior to execution of 'utils.eda'; this may result in unpredictable behaviour

ELEC 475: EXPLORATORY DATA ANALYSIS - DATASET: CIFAR10

[Dataset Inventory & Technical Profile]
  Dataset Name              : CIFAR10
  Total Training Samples    : 50,000
  Number of Classes (C)     : 10
  Native Image Resolution   : 32 x 32 x 3
  Standardized Model Shape  : 3 x 64 x 64 (Tensor: float32, range [0.0, 1.0])
  Approximate Disk Storage  : 177.6 MB

[Pillar 1: Class Balance, Frequency Profiling & Imbalance Diagnostics]
  Samples Per Class (Mean)  : 5000.0 (Median: 5000.0, Std: 0.00)
  Class Range [Min, Max]    : [5000, 5000]
  Imbalance Ratio (IR)      : 1.00
  Shannon Entropy (H)       : 3.3219 bits (Max possible: 3.3219 bits)
  Entropy Ratio (H / H_max) : 1.0000 (1.0 = perfectly uniform distribution)
  Gini Impurity Index       : 0.9000
  Per-Class Distribution Breakdown:
    Class 00 (airp

In [ ]:
!python train.py --model lenet --dataset tiny_imagenet --epochs 10 --no-augment --seed 42 --save-model checkpoints/lenet_tiny.pth --plot-file plots/loss_lenet_tiny.png 2>&1 | tee results/lenet_tiny.txt
!python train.py --model lenet_modern --dataset tiny_imagenet --epochs 10 --no-augment --seed 42 --save-model checkpoints/lenet_modern_tiny.pth --plot-file plots/loss_lenet_modern_tiny.png 2>&1 | tee results/lenet_modern_tiny.txt
!python train.py --model alexnet --norm none --dataset tiny_imagenet --epochs 10 --no-augment --seed 42 --save-model checkpoints/alexnet_none_tiny.pth --plot-file plots/loss_alexnet_none_tiny.png 2>&1 | tee results/alexnet_none_tiny.txt
!python train.py --model alexnet --norm lrn --dataset tiny_imagenet --epochs 10 --no-augment --seed 42 --save-model checkpoints/alexnet_lrn_tiny.pth --plot-file plots/loss_alexnet_lrn_tiny.png 2>&1 | tee results/alexnet_lrn_tiny.txt
!python train.py --model alexnet --norm bn --dataset tiny_imagenet --epochs 10 --no-augment --seed 42 --save-model checkpoints/alexnet_bn_tiny.pth --plot-file plots/loss_alexnet_bn_tiny.png 2>&1 | tee results/alexnet_bn_tiny.txt
!python train.py --model zoo_alexnet --dataset tiny_imagenet --epochs 10 --no-augment --seed 42 --save-model checkpoints/zoo_alexnet_tiny.pth --plot-file plots/loss_zoo_alexnet_tiny.png 2>&1 | tee results/zoo_alexnet_tiny.txt

ELEC 475: Lab 2 Training Engine
Device        : cuda (Tesla T4)
Model         : LENET (Norm: N/A)
Dataset       : TINY_IMAGENET (Augmentation: False)
Hyperparams   : Epochs=10, BatchSize=128, LR=0.001, Seed=42
Early Stopping: Disabled

Initializing DataLoaders...
Train batches: 781 (100000 samples), Val batches: 79, Classes: 200
Computational Complexity Profile: LENET (bn)
Input Tensor Shape   : (1, 3, 64, 64)
------------------------------------------------------------
Total Parameters     : 354,636 (0.3546 M)
Trainable Parameters : 354,636
Multiply-Accumulates : 3,601,864 (3.60 MMACs)
Floating Point Ops   : 7,203,728 (7.20 MFLOPs)
Weight Size (FP32)   : 1.35 MB

Detailed Layer Specification:
Layer (type:depth-idx)                   Output Shape              Param #
LeNet5                                   [1, 200]                  --
├─Sequential: 1-1                        [1, 16, 13, 13]           --
│    └─Conv2d: 2-1                       [1, 6, 60, 60]            456
│    └─Tanh